In [ ]:
import json
import os
import random
import cv2
import matplotlib.pyplot as plt

# 1. Configure dataset paths
base_dir = r"D:\01_University_&_Study\Samsung Innovation Campus - Immersive with MiSK Skills\CapstoneProject\MarineGuard_Datasets\SeaClear\Seaclear Marine Debris Dataset"
json_path = os.path.join(base_dir, "dataset.json")

# 2. Build a recursive lookup map for all images across site subfolders (e.g., Bistrina, Slano)
image_path_map = {}
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_path_map[file] = os.path.join(root, file)

print(f"Successfully indexed {len(image_path_map)} images across subfolders.")

# 3. Load COCO annotations and filter target classes (Nets and Ropes)
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

target_names = {'net_plastic': 'Net', 'rope_fiber': 'Rope', 'rope_plastic': 'Rope'}
target_cat_ids = {cat['id']: target_names[cat['name']] for cat in data['categories'] if cat['name'] in target_names}

img_to_anns = {}
for ann in data['annotations']:
    if ann['category_id'] in target_cat_ids:
        img_to_anns.setdefault(ann['image_id'], []).append(ann)

# 4. Sample 4 random annotated images for visual verification
valid_img_ids = list(img_to_anns.keys())
sample_ids = random.sample(valid_img_ids, min(4, len(valid_img_ids)))
images_info = {img['id']: img for img in data['images']}

# 5. Render images with ground-truth bounding boxes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, img_id in enumerate(sample_ids):
    img_data = images_info[img_id]
    file_name = os.path.basename(img_data['file_name'])
    img_path = image_path_map.get(file_name)
    
    if not img_path:
        print(f"Image not found: {file_name}")
        continue
        
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    for ann in img_to_anns[img_id]:
        x, y, w, h = [int(v) for v in ann['bbox']]
        label = target_cat_ids[ann['category_id']]
        color = (255, 0, 0) if label == 'Net' else (0, 255, 0)  # Red for Net, Green for Rope
        
        cv2.rectangle(image, (x, y), (x + w, y + h), color, 3)
        cv2.putText(image, label, (x, max(y - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
        
    axes[idx].imshow(image)
    axes[idx].set_title(f"File: {file_name} | Site: {os.path.basename(os.path.dirname(img_path))}")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()